# MAI-AHLT - NERC
### Josep de Cid | Victor Giménez

In [1]:
import argparse
import os
import re
import string

import pycrfsuite

from glob import glob
from typing import List, Tuple, Dict, Set, Union
from collections import Counter, defaultdict
from datetime import datetime

from bs4 import BeautifulSoup as Bs
from bs4.element import Tag
from nltk import word_tokenize
from nltk.corpus import stopwords
from tqdm import tqdm

In [2]:
# Change directory to read datasets, JAR... properly (in our local folder structure)
os.chdir(os.path.join('..', '..'))

STOPWORDS = set(stopwords.words('english'))

TYPE_WORDS = {}
COMMON_SUFFIXES = {}

CRF_MODEL_NAME = os.path.join('labs', 's2', 'crf.model')

HSDB = set()
DRUG_BANK = dict(drug=set(), drug_n=set(), group=set(), brand=set())

EVAL_JAR_DIR = os.path.join('project', 'eval', 'evaluateNER.jar')

In [3]:
def evaluate(input_dir: str, output_dir: str) -> None:
    print(os.popen(f'java -jar {EVAL_JAR_DIR} {input_dir} {output_dir}').read())

## Task 1

For both goals of task 1 (with and without using external information) we use the same code. The difference is that, in the case of goal 1, variable TYPE_WORDS has no external information (as it didn't exist). This can be seen in the extract_data_information method, which checks whether our goal is 2 for checking the HSDB and DrugBank datasets.

We describe the logic of the rule based system in steps:
1. If the token is a drug common to the English lexicon, we mark it as such. This is because initially we had some tests with checking if the word was a word of English, which lowered our accuracy. It was removed, but this rule heightens our accuracy somewhat.
2. If the token is a stopword or a punctuation mark, we ignore it, for efficiency purposes.
3. If the token appears in the training 'group' category, we mark it as such.
4. If the token has a dash and is uppercase, it is oftentimes a drug\_n, even if it is only a single drug. The same if it has a combination of some characters like PGE2 or similar.
5. Tokens starting with alpha- or similar are drug\_n
6. Any other token being uppercase is a brand.
7. We check whether the token has a common suffix of a category (extracted from train), and mark it as such (except for drug\_n, as it is very like drug).
8. We repeat step 3 with drugs
9. For multi-group structures (extracted only for goal2) we mark them as such with a special type of mark.

After that is done, we attempt to merge the tokens extracted greedily: after finding a class or m-class in the sequence, we greedily merge with the ones afterward.

In [4]:
def tokenize(sentence: str) -> List[Tuple[str, int, int]]:
    """
    Tokenize a sentence to obtain each token and its start and end position inside the sentence.

    Parameters
    ----------
    sentence : str
        Plain text sentence.

    Returns
    -------
    List[Tuple[str, int, int]]
        List of tokens that constitute the sentence with the associated start and end position.
    """
    tokenized_output = []
    tokenized_words = word_tokenize(sentence)

    i = 0
    for word in tokenized_words:
        word_pos = sentence.find(word, i)
        tokenized_output.append((word, word_pos, word_pos + len(word) - 1))
        i = word_pos + len(word)

    return tokenized_output

In [5]:
def extract_grouped_entities(input_dir: str) -> Dict[str, List[str]]:
    """
    Exrtract grouped entities by type = {'drug', 'drug_n', 'group', 'brand'}
    
    Parameters
    ----------
    input_dir : str
        Directory for the input data.
        
    Returns
    -------
    Dict[str, List[str]]
        List of entities grouped by common key type.
    """
    options = ['drug', 'drug_n', 'brand', 'group']
    grouped_entities = {k: [] for k in options}

    for file in tqdm(glob(os.path.join(input_dir, '*')), desc='Reading files', ncols=100):
        with open(file, mode='r') as f:
            content = f.read().strip()
            bs_content = Bs(content, 'lxml')
            entities = bs_content.find_all('entity')

            for entity in entities:
                if entity['type'] in options:
                    grouped_entities[entity['type']].append(entity['text'])

    return grouped_entities

In [6]:
def extract_most_common_suffixes(grouped_entities: Dict[str, List[str]]) -> Dict[str, Set[str]]:
    """
    Extract most common suffixes from the given grouped entities.

    Parameters
    ----------
    grouped_entities : Dict[str, List[str]]
        Entities grouped by type (`drug`, `drug_n`, `brand` or `group`).

    Returns
    -------
    Dict[str, Set[str]]
        Most common suffixes grouped by type analogously to the input format.
    """

    def most_common_suffixes_of_length(n: int) -> List[Tuple[str, int]]:
        filtered_entities = filter(lambda x: ' ' not in x, set(grouped_entities[d_type]))
        suffixes = list(map(lambda x: x[-n:], set(filtered_entities)))
        counter = Counter(suffixes)
        return counter.most_common()

    most_common_suffixes = defaultdict(set)
    for d_type, suf_len in [('drug', 5), ('drug_n', 4), ('group', 4), ('brand', 5)]:
        # Extract most common suffixes of the given length for the corresponding type.
        for w, _ in most_common_suffixes_of_length(suf_len):
            most_common_suffixes[d_type].add(w)

    return most_common_suffixes

In [7]:
def extract_entities(tokenized_sentence: List[Tuple[str, int, int]], sentence: str) -> List[Dict[str, str]]:
    """
    Extract entities applying a list of rules to the tokenized sentence.

    Parameters
    ----------
    tokenized_sentence : List[Tuple[str, int, int]]
         List of tokens that constitute the sentence with the associated start and end position.
    sentence : str
        Plain text sentence.

    Returns
    -------
    List[Dict[str, str]]
        List of entities for the given sentence, where the entity is described with a dictionary with the following
        format: {'name': 'test', 'offset': '0-3', 'type': 'drug'}. Possible types: ['drug', 'drug_n', 'group', 'brand'].
    """
    extracted_entities = []
    marked_iob = []

    def mark_as_(d_type: str = None):
        marked_iob.append(d_type)

    def output_as_(d_type: str, name: str, offset: str):
        drug_info = {'name': name, 'offset': offset, 'type': d_type}
        extracted_entities.append(drug_info)

    def detect_multi(acceptable_tags, tag_to_mark):
        if tag in acceptable_tags:
            idx_g = 1
            if idx + idx_g < len(marked_iob) and marked_iob[idx + idx_g] in acceptable_tags:
                marked_iob[idx] = None
                while idx + idx_g < len(marked_iob) and marked_iob[idx + idx_g] in acceptable_tags:
                    marked_iob[idx + idx_g] = None
                    idx_g += 1

                end = tokenized_sentence[idx + idx_g - 1][2]
                group_sentence = sentence[start:end]
                output_as_(tag_to_mark, group_sentence, f'{start}-{end}')

    for idx, (token, _, _) in enumerate(tokenized_sentence):
        if token.lower() in ['cannabis', 'cocaine', 'amphetamine', 'nitrites', 'cocaine',
                             'ecstasy', 'lsd', 'heroin', 'crack', 'lithium']:
            mark_as_('drug')
        elif token in STOPWORDS or token in string.punctuation or token in ["''", "``", '""']:
            mark_as_(None)
        elif token in TYPE_WORDS['group']:
            mark_as_('group')
        elif token.isupper() and '-' in token:
            mark_as_('drug_n')
        elif any(map(lambda x: x in token, ['PGE2', 'CMC', 'EFG', '(V)', '(-)'])):
            mark_as_('drug_n')
        elif token.lower().startswith(('alpha-', 'beta-')):
            mark_as_('drug_n')
        elif token.isupper():
            mark_as_('brand')

        # Common suffixes
        elif any(map(lambda x: token.endswith(x), COMMON_SUFFIXES['drug'])):
            mark_as_('drug' if '-' not in token else 'drug_n')
        elif any(map(lambda x: token.endswith(x), COMMON_SUFFIXES['group'])):
            mark_as_('group')
        elif any(map(lambda x: token.endswith(x), COMMON_SUFFIXES['brand'])):
            mark_as_('brand')

        # Matching words by type
        elif token in TYPE_WORDS['drug']:
            mark_as_('drug')
        elif token in TYPE_WORDS.get('multi-group', []):
            mark_as_('m-group')
        elif token in TYPE_WORDS.get('multi-drug', []):
            mark_as_('drug_n')
        elif token in TYPE_WORDS.get('multi-brand', []):
            mark_as_('m-brand')

        # By default, mark anything else as `None` to filter them out when parsing together multi-tokens.
        else:
            mark_as_(None)

    # Multi-token

    for idx, (tag, (token, start, end)) in enumerate(zip(marked_iob, tokenized_sentence)):
        detect_multi(['group', 'm-group'], 'group')
        detect_multi(['drug', 'drug_n'], 'drug_n')
        detect_multi(['brand', 'm-brand'], 'brand')

    for tag, (token, start, end) in zip(marked_iob, tokenized_sentence):
        if tag is not None and tag != 'm-group':
            interval = f'{start}-{end}'
            output_as_(tag, token, interval)

    return extracted_entities

In [8]:
def output_entities(s_id: str, entities: List[Dict[str, str]], output_dir: str) -> None:
    """
    Outputs entities of a sentence with the following format: `{s_id}|{offset}|{token}|{category}`

    Parameters
    ----------
    s_id : str
        Sentence identifier.
    entities : List[Dict[str, str]]
        List of entities for the given sentence, where the entity is described with a dictionary with the following
        format: {'name': 'test', 'offset': '0-3', 'type': 'drug'}. Possible types: ['drug', 'drug_n', 'group', 'brand'].
    output_dir : str
        File path to print the output to.
    """
    output_data = []
    for entity in entities:
        output_data.append(f'{s_id}|{entity["offset"]}|{entity["name"]}|{entity["type"]}')

    output_data = '\n'.join(output_data)
    if len(output_data) > 0:
        output_data += '\n'

    with open(output_dir, mode='a') as f:
        f.write(output_data)

In [9]:
def extract_data_information(input_dir: str, goal: int):
    """
    Extract data information from external folders
    
    Parameters
    ----------
    input_dir : str
        Directory for the input data.
        
    goal : int
        The corresponding goal of the task, as some information is only loaded for the second task.
    """
    global TYPE_WORDS
    TYPE_WORDS = extract_grouped_entities(input_dir)
    TYPE_WORDS['multi-group'] = word_tokenize(' '.join(TYPE_WORDS['group']))

    global COMMON_SUFFIXES
    COMMON_SUFFIXES = extract_most_common_suffixes(TYPE_WORDS)

    if goal == 2:
        # For the second goal, we extract information from external knowledge sources such as:
        # - HSDB (Hazardous Substances Data Bank)
        # - Drug Bank
        with open(os.path.join('project', 'resources', 'HSDB.txt'), mode='r') as f:
            TYPE_WORDS['drug'] += f.read().split('\n')

        with open(os.path.join('project', 'resources', 'DrugBank.txt'), mode='r') as f:
            for line in f.readlines():
                sentence, token_type = line.strip().split('|')
                tokenized_sentence = tokenize(sentence)
                if len(tokenized_sentence) > 1:
                    if f'multi-{token_type}' not in TYPE_WORDS.keys():
                        TYPE_WORDS[f'multi-{token_type}'] = []
                    TYPE_WORDS[f'multi-{token_type}'] += tokenized_sentence
                else:
                    TYPE_WORDS[token_type] += sentence

    for k, v in COMMON_SUFFIXES.items():
        COMMON_SUFFIXES[k] = set(v)

In [10]:
def nerc(input_dir: str, input_dir_eval: str, goal: int, output_dir: str) -> None:
    # Clean output dir
    open(output_dir, mode='w').close()
    
    extract_data_information(input_dir, goal)
    for file in tqdm(glob(os.path.join(input_dir_eval, '*')), ncols=100, desc='Detecting NERC'):
        with open(file, mode='r') as f:
            content = f.read().strip()
            bs_content = Bs(content, 'lxml')
            sentences = bs_content.find_all('sentence')
            for sentence in sentences:
                sid, text = sentence['id'], sentence['text']
                token_list = tokenize(text)
                entities = extract_entities(token_list, text)
                output_entities(sid, entities, output_dir)
    evaluate(input_dir_eval, output_dir)

### Goal 1 - Devel

In [11]:
nerc('project/data/Train', 'project/data/Devel', 1, 'labs/s1/task9.1_VJ_42.txt')

Detecting NERC: 100%|███████████████████████████████████████████████| 85/85 [00:03<00:00, 28.06it/s]


Gold project/data/Devel
Submission  labs/s1/task9.1_VJ_42.txt
Directory gold project/data/Devel
[transDirXMLToMapEntities] dir:project/data/Devel
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=681, entities: 1771
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
1297	449	0	25	756	1771	0.52	0.73	0.61



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
1407	339	0	25	756	1771	0.56	0.79	0.66



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
1407	0	339	25	756	1771	0.56	0.89	0.69



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
1480	266	0	25	756	1771	0.59	0.84	0.69




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no i

### Goal 1 - Test

In [12]:
nerc('project/data/Train', 'project/data/Test-NER', 1, 'labs/s1/task9.1_VJ_42.txt')

Detecting NERC: 100%|█████████████████████████████████████████████| 112/112 [00:02<00:00, 39.16it/s]


Gold project/data/Test-NER
Submission  labs/s1/task9.1_VJ_42.txt
Directory gold project/data/Test-NER
[transDirXMLToMapEntities] dir:project/data/Test-NER
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=324, entities: 686
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
376	191	0	119	513	686	0.35	0.55	0.43



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
427	140	0	119	513	686	0.4	0.62	0.48



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
427	0	140	119	513	686	0.4	0.72	0.51



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
443	124	0	119	513	686	0.41	0.65	0.5




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no 

The end results, show a good performance for Devel data but not that good for Test with all classes with the exception of drug\_n. In particular, for drug\_n, we observe that both precision and recall metrics are very low. We argue that this is due to the complexity of the possible drug_n NE, which usually consist in multi-token combinations and are much more variable and difficult to extract with hand-crafted rules.

### Goal 2 - Devel

In [13]:
nerc('project/data/Train', 'project/data/Devel', 2, 'labs/s1/task9.1_VJ_42.txt')

Detecting NERC: 100%|███████████████████████████████████████████████| 85/85 [01:18<00:00,  1.09it/s]


Gold project/data/Devel
Submission  labs/s1/task9.1_VJ_42.txt
Directory gold project/data/Devel
[transDirXMLToMapEntities] dir:project/data/Devel
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=681, entities: 1771
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
1320	399	0	52	811	1771	0.52	0.75	0.61



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
1431	288	0	52	811	1771	0.57	0.81	0.67



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
1431	0	288	52	811	1771	0.57	0.89	0.69



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
1480	239	0	52	811	1771	0.58	0.84	0.69




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no i

### Goal 2 - Test

In [14]:
nerc('project/data/Train', 'project/data/Test-NER', 2, 'labs/s1/task9.1_VJ_42.txt')

Detecting NERC: 100%|█████████████████████████████████████████████| 112/112 [01:24<00:00,  1.33it/s]


Gold project/data/Test-NER
Submission  labs/s1/task9.1_VJ_42.txt
Directory gold project/data/Test-NER
[transDirXMLToMapEntities] dir:project/data/Test-NER
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=324, entities: 686
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
385	175	0	126	563	686	0.34	0.56	0.43



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
436	124	0	126	563	686	0.39	0.64	0.48



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
436	0	124	126	563	686	0.39	0.73	0.51



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
442	118	0	126	563	686	0.39	0.64	0.49




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are 

As we can see, adding external information doesn't really help to get a better performance, just a slightly improvement (but non-significant) for drug and brand in the Devel data. The problematic class that lowers the score is still drug_n, and again, we can argue that the lack of improvement is due to the complexity of this class. However we are able to improve the score substantially using ML methods, as we show in Task 2.

## Task 2

The following is the code for task 2, shared between the two goals. As before, the only difference between goals is the value of the goal flag. We only check external information if the goal flag is 5, so the features related to HSDB and Drug-Bank can only appear when the flag is 5.

The features we consider are the following:

* Bias (common to all features)
* Form (token in lowercase)
* Length 4 prefix and suffix
* Next word (in lowercase)
* Previous word (in lowercase)
* Whether the token is all uppercase, the first letter is uppercase, the token is all lowercase, or otherwise.
* Whether the word has a hyphen (and other punctuation), whether it is pure punctuation, it is a number, it is fully alphabetic or it is alphanumeric.
* The length of the token (less than 5, less than 10, less than 15 or otherwise)
* (For task 5) whether the token is in HSDB (we tokenise the entries in the dataset)
* (For task 5) whether the token is in Drug\_Bank and the class assigned there. (we tokenise the entries in the dataset)

To merge the IOB tags, what we do is: if are still inside a tagging (matching BI*) and we receive B or O, we dump the entity so far. Otherwise, the tag starts with I and we merge it. Independently, if we receive a B, then we start accumulating a new entity tagging.

In [15]:
def extract_features(tokenized_sentence: List[Tuple[str, int, int]]) -> List[List[str]]:
    """
    Extract features from a tokenized sentence to fit a ML classifier model.

    Parameters
    ----------
    tokenized_sentence : List[Tuple[str, int, int]]
        List of tuples containing each token information formatted as <Token, start_idx, end_idx>

    Returns
    -------
    List[List[str]]
        A list of binary features (represented as strings) for each token of the sentence.
    """
    features = []
    for idx, (token, offset_from, offset_to) in enumerate(tokenized_sentence):
        # Token, suffixes, and previous + next word
        token_features = ['bias',
                          f'form={token.lower()}',
                          f'pre4={token[:4].lower()}',
                          f'suf4={token[-4:].lower()}',
                          f'next={tokenized_sentence[idx + 1][0].lower() if idx + 1 < len(tokenized_sentence) else "_EoS_"}',
                          f'prev={tokenized_sentence[idx - 1][0].lower() if idx - 1 >= 0 else "_BoS_"}']

        # Case (Upper, Capitalized, Lower or Mixed)
        if token.isupper():
            token_features.append('upper')
        elif token[0].isupper():
            token_features.append('capitalized')
        elif token.islower():
            token_features.append('lower')
        elif token.isalnum():
            token_features.append('mixed-case')

        # Type of characters (punctuation, alphanumeric, digit or containing hyphens
        if '-' in token:
            if string.ascii_lowercase in token.lower() and string.punctuation in token:
                token_features.append('mixed-hyphen')
            token_features.append('hyphen')
        elif token in string.punctuation or token in ["''", "``", '""']:
            token_features.append('punctuation')
        elif token.isdigit():
            token_features.append('numeric')
        elif token.isalpha():
            token_features.append('alpha')
        else:
            token_features.append('alphanumeric')

        # Quantized token length into 4 bins (<5, <10, <15, >=15)
        if len(token) < 5:
            token_features.append(f'len=<5')
        elif len(token) < 10:
            token_features.append(f'len=<10')
        elif len(token) < 15:
            token_features.append(f'len=<15')
        else:
            token_features.append(f'len=>=15')

        if token.lower() in HSDB:
            token_features.append('HSDB')

        if token.lower() in DRUG_BANK['drug']:
            token_features.append('DRUG_BANK_DRUG')
        elif token.lower() in DRUG_BANK['drug_n']:
            token_features.append('DRUG_BANK_DRUG_N')
        elif token.lower() in DRUG_BANK['brand']:
            token_features.append('DRUG_BANK_BRAND')
        elif token.lower() in DRUG_BANK['group']:
            token_features.append('DRUG_BANK_GROUP')

        features.append(token_features)

    return features

In [16]:
def output_features(s_id: str, tokenized_sentence: List[Tuple[str, int, int, str]], features: List[List[str]],
                    output_dir: str = None) -> None:
    """
    Print to a file or stdout the features vectors for a sentence.

    Parameters
    ----------
    s_id : str
        Sentence identifier.
    tokenized_sentence : List[Tuple[str, int, int, str]]
        List of tuples containing each token information formatted as <Token, start_idx, end_idx, gold_class>
    features : List[List[str]]
        List of binary features (represented as strings) for each token of the sentence.
    output_dir : str, optional, default=None
        File path to print the output to. If set as `None` (default behavior) it prints to the stdout.
    """
    output_data = []
    for token_info, token_features in zip(tokenized_sentence, features):
        output = [s_id, *[str(x) for x in token_info], *token_features]
        output_data.append('\t'.join(output))

    if output_dir is None:
        print('\n'.join(output_data) + '\n')
    else:
        with open(output_dir, mode='a') as f:
            f.write('\n'.join(output_data) + '\n')

In [17]:
def parse_gold_sentence_entities(tokenized_sentence: List[Tuple[str, int, int]],
                                 sentence: Tag, is_eval: bool = False) -> List[Tuple[str, int, int, Union[str, None]]]:
    """
    Parse gold class from the training XML data in IOB format.

    Parameters
    ----------
    tokenized_sentence : List[Tuple[str, int, int]]
        List of tuples containing each token information formatted as <Token, start_idx, end_idx>
    sentence : bs4.element.Tag
        XML formatted sentence data.
    is_eval : bool, optional, default=False

    Returns
    -------
    List[Tuple[str, int, int, str]]
        Extended tokenized_sentence containing the gold class tag in IOB format as the last element in the tuple.
    """
    # In evaluation mode we don't have access to real entities tags.
    # Hence, we just set them as `O` tags.
    if is_eval:
        return [(token, start, end, None) for token, start, end in tokenized_sentence]

    entities = sentence.find_all('entity')

    if len(entities) == 0:
        return [(token, start, end, 'O') for token, start, end in tokenized_sentence]

    def parse_entity(entity) -> Tuple[str, int, int]:
        offsets = list(map(int, re.split(r'[-;]', entity['charoffset'])))
        return entity['type'], int(offsets[0]), int(offsets[-1])

    def generate_iob(tag: str, type: str = None):
        return token, start, end, 'O' if tag == 'O' else f'{tag}-{type}'

    entities: List[Tuple[str, int, int]] = [parse_entity(e) for e in entities]

    idx_e = 0
    e_type, e_start, e_end = entities[idx_e]

    is_inside = False
    end_entities = False

    output_token_list = []
    for token, start, end in tokenized_sentence:
        # Evaluate the following obtained entity.
        if not end_entities and start > e_end:
            is_inside = False
            idx_e += 1
            if idx_e >= len(entities):
                end_entities = True
            else:
                e_type, e_start, e_end = entities[idx_e]

        # If last sentence has been already added, mark all the following tokens as `O`.
        if end_entities:
            output_token_list.append(generate_iob('O'))

        # If the entity start corresponds to the token start mark it as `B`.
        elif start == e_start:
            output_token_list.append(generate_iob('B', e_type))
            is_inside = True

        # If the token is between the entity start and end mark it as `B` or `I` depending on there is a previous `B`.
        elif start >= e_start and end <= e_end:
            iob_tag = 'I' if is_inside else 'B'
            output_token_list.append(generate_iob(iob_tag, e_type))

        # Otherwise, mark it as `O` and discard any possible immediate `I`.
        else:
            output_token_list.append(generate_iob('O'))
            is_inside = False

    return output_token_list

In [18]:
def output_entities(s_id: str, tokenized_sentence: List[Tuple[str, int, int, Union[str, None]]], predictions: List[str],
                    output_dir: str) -> None:
    """
    Outputs entities of a sentence with the following format: `{s_id}|{offset}|{token}|{category}`

    Parameters
    ----------
    s_id : str
        Sentence identifier.
    tokenized_sentence : List[str, int, int, Union[str, None]]
        List of tuples containing each token information formatted as <Token, start_idx, end_idx, gold_class>
        `gold_class` is None when dealing with evaluation dataset.
    predictions : List[str]
        List of predictions for the sentence in IOB format.
    output_dir : str
        File path to print the output to.
    """
    output_data = []
    c_entity = None

    for (token, start, end, _), prediction in zip(tokenized_sentence, predictions):
        if c_entity is not None:
            if prediction.startswith('I-'):
                c_entity = (f'{c_entity[0]} {token}', c_entity[1], end, c_entity[3])
            else:
                offset = f'{c_entity[1]}-{c_entity[2]}'
                output_data.append(f'{s_id}|{offset}|{c_entity[0]}|{c_entity[3]}')
                c_entity = None

        if prediction.startswith('B-'):
            c_entity = (token, start, end, prediction[2:])

    if c_entity is not None:
        offset = f'{c_entity[1]}-{c_entity[2]}'
        output_data.append(f'{s_id}|{offset}|{c_entity[0]}|{c_entity[3]}')

    output_data = '\n'.join(output_data)
    if len(output_data) > 0:
        output_data += '\n'

    with open(output_dir, mode='a') as f:
        f.write(output_data)

In [19]:
def run_phase(mode: str, input_dir: str, output_dir: str) -> None:
    """
    Run either training or test for the CRF model.
    
    Parameters
    ----------
    mode : str
        Either train or test.
    
    output_dir : str
        Directory for the output data.
    """
    if mode == 'train':
        model = pycrfsuite.Trainer(verbose=False)
        model.set_params({
            'c1': 0.1,  # L1 penalty
            'c2': 0.01,  # L2 penalty
            'max_iterations': 200,  # maximum number of iterations
            'feature.possible_transitions': True  # whether to include transitions that are possible, but not observed
        })
    else:
        model = pycrfsuite.Tagger()
        model.open(CRF_MODEL_NAME)

    pb_desc = 'Parsing training data' if mode == 'train' else 'Predicting entities'
    for file in tqdm(glob(os.path.join(input_dir, '*')), ncols=100, desc=pb_desc):
        output_features_path = os.path.sep.join(output_dir.split(os.path.sep)[:-1])

        with open(file, mode='r') as f:
            content = f.read().strip()
            bs_content = Bs(content, 'lxml')
            sentences = bs_content.find_all('sentence')
            for sentence in sentences:
                sid, text = sentence['id'], sentence['text']
                tokens_list = tokenize(text)

                features = extract_features(tokens_list)

                tokens_list = parse_gold_sentence_entities(tokens_list, sentence, is_eval=mode != 'train')
                output_features(sid, tokens_list, features, os.path.join(output_features_path, f'features_{mode}.txt'))

                if mode == 'train':
                    model.append(features, list(map(lambda x: x[3], tokens_list)))
                else:
                    prediction = model.tag(features)
                    output_entities(sid, tokens_list, prediction, output_dir)

    if mode == 'train':
        print('Training CRF model...')
        start_time = datetime.now()
        model.train(CRF_MODEL_NAME)
        print(f'Model trained in {(datetime.now() - start_time).seconds}s')

In [20]:
def extract_external_information():
    global HSDB
    with open(os.path.join('project', 'resources', 'HSDB.txt'), mode='r') as f:
        lines = f.read().strip().split('\n')
        for entry in lines:
            tokens = tokenize(entry)
            for token, _, _ in tokens:
                HSDB.add(token.lower())

    global DRUG_BANK
    with open(os.path.join('project', 'resources', 'DrugBank.txt'), mode='r') as f:
        lines = f.read().strip().split('\n')
        for line in lines:
            entry, entry_type = line.strip().split('|')
            tokens = tokenize(entry)
            for token, _, _ in tokens:
                DRUG_BANK[entry_type].add(token.lower())

In [21]:
def nerc(input_dir: str, input_dir_eval: str, goal: int, output_dir: str) -> None:
    open(output_dir, mode='w').close()
    if goal == 4:
        extract_external_information()
    run_phase('train', input_dir, output_dir)
    run_phase('eval', input_dir_eval, output_dir)
    evaluate(input_dir_eval, output_dir)

### Goal 3 - Devel

In [22]:
nerc('project/data/Train', 'project/data/Devel', 3, 'labs/s2/task9.1_VJ_42.txt')

Parsing training data: 100%|█████████████████████████████████████| 629/629 [00:05<00:00, 115.00it/s]


Training CRF model...


Predicting entities:  15%|██████▎                                  | 13/85 [00:00<00:00, 127.99it/s]

Model trained in 17s


Predicting entities: 100%|██████████████████████████████████████████| 85/85 [00:01<00:00, 68.27it/s]


Gold project/data/Devel
Submission  labs/s2/task9.1_VJ_42.txt
Directory gold project/data/Devel
[transDirXMLToMapEntities] dir:project/data/Devel
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=681, entities: 1771
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
1444	129	0	198	45	1771	0.89	0.82	0.85



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
1485	88	0	198	45	1771	0.92	0.84	0.88



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
1485	0	88	198	45	1771	0.92	0.86	0.89



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
1523	50	0	198	45	1771	0.94	0.86	0.9




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no inclu

### Goal 3 - Test

In [23]:
nerc('project/data/Train', 'project/data/Test-NER', 3, 'labs/s2/task9.1_VJ_42.txt')

Parsing training data: 100%|█████████████████████████████████████| 629/629 [00:06<00:00, 104.56it/s]


Training CRF model...


Predicting entities:  21%|████████▌                               | 24/112 [00:00<00:00, 204.36it/s]

Model trained in 17s


Predicting entities: 100%|███████████████████████████████████████| 112/112 [00:00<00:00, 228.97it/s]


Gold project/data/Test-NER
Submission  labs/s2/task9.1_VJ_42.txt
Directory gold project/data/Test-NER
[transDirXMLToMapEntities] dir:project/data/Test-NER
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=324, entities: 686
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
442	73	0	171	37	686	0.8	0.64	0.71



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
476	39	0	171	37	686	0.86	0.69	0.77



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
476	0	39	171	37	686	0.86	0.72	0.79



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
462	53	0	171	37	686	0.84	0.67	0.75




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no includ

The end results, both for the validation and the test sets show a good performance for all classes with the exception of drug\_n. In particular, for drug\_n, we observe a high imbalance between the precision and recall metrics (very high precision and low recall). We argue that this is due to the dissimilarity of the test/devel sets with respect to the train set, as in our posterior analysis we saw many entities of type drug\_n which were not seen in the training set, and which we had a hard time differentiating from entities of the drug category.

### Goal 4 - Devel

In [24]:
nerc('project/data/Train', 'project/data/Devel', 4, 'labs/s2/task9.1_VJ_42.txt')

Parsing training data: 100%|█████████████████████████████████████| 629/629 [00:04<00:00, 135.71it/s]


Training CRF model...


Predicting entities:  18%|███████▏                                 | 15/85 [00:00<00:00, 138.98it/s]

Model trained in 18s


Predicting entities: 100%|█████████████████████████████████████████| 85/85 [00:00<00:00, 127.26it/s]


Gold project/data/Devel
Submission  labs/s2/task9.1_VJ_42.txt
Directory gold project/data/Devel
[transDirXMLToMapEntities] dir:project/data/Devel
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=681, entities: 1771
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
1485	113	0	173	46	1771	0.9	0.84	0.87



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
1513	85	0	173	46	1771	0.92	0.85	0.89



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
1513	0	85	173	46	1771	0.92	0.88	0.9



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
1565	33	0	173	46	1771	0.95	0.88	0.92




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no includ

### Goal 4 - Test

In [25]:
nerc('project/data/Train', 'project/data/Test-NER', 4, 'labs/s2/task9.1_VJ_42.txt')

Parsing training data: 100%|█████████████████████████████████████| 629/629 [00:04<00:00, 133.33it/s]


Training CRF model...


Predicting entities:  21%|████████▌                               | 24/112 [00:00<00:00, 237.19it/s]

Model trained in 17s


Predicting entities: 100%|███████████████████████████████████████| 112/112 [00:00<00:00, 253.58it/s]


Gold project/data/Test-NER
Submission  labs/s2/task9.1_VJ_42.txt
Directory gold project/data/Test-NER
[transDirXMLToMapEntities] dir:project/data/Test-NER
 Gold standard saved in goldNER.txt

Gold loaded. Sentences=324, entities: 686
task9.1_VJ_42_scores.log created...
SCORES FOR THE GROUP: VJ RUN=42

Warning!!!! some sentences are no included in the gold!!!

Strict matching (boundaries + type)
cor	inc	par	mis	spu	total	prec	recall	F1
479	73	0	134	32	686	0.82	0.7	0.75



Warning!!!! some sentences are no included in the gold!!!

Exact matching
cor	inc	par	mis	spu	total	prec	recall	F1
511	41	0	134	32	686	0.88	0.74	0.8



Warning!!!! some sentences are no included in the gold!!!

Partial matching
cor	inc	par	mis	spu	total	prec	recall	F1
511	0	41	134	32	686	0.88	0.77	0.82



Warning!!!! some sentences are no included in the gold!!!

type matching
cor	inc	par	mis	spu	total	prec	recall	F1
502	50	0	134	32	686	0.86	0.73	0.79




SCORES FOR ENTITY TYPE
Warning!!!! some sentences are no include

As we can see, our results for both the validation and the test set are generally good for brand, group and drug. We notice that there is no clear correlation between precision and recall (both are balanced) as we should expect to maximise F1. However, we see an extremely bad result in the recall of drug\_n. Looking at the validation and test set (after confirming these were our final predictions so as not to influence our analysis) we can argue that it is extremely difficult to detect most of the cases of drug\_n class, as the distribution varies too significantly with respect to the training set. Most of the entities are single-word, previously unseen words, which we could easily mistake with drug. We also observe a small drop in F1 score in this class with respect to the version in goal 3 (no external data), possibly due to the model overfitting to the external data information as it works for the rest of the cases.